In [ ]:
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

OUT = Path("../data/processed/simplewiki")
pages = pq.read_table(OUT / "pages.parquet")
edges = pq.read_table(OUT / "edges.parquet")

page_ids = pages["page_id"].to_numpy()
src = edges["source"].to_numpy()
tgt = edges["target"].to_numpy()

print(f"articles: {len(page_ids):,}   edges: {len(src):,}")
print(f"largest page_id: {page_ids.max():,}")
print(f"edges already sorted by source? {bool(np.all(src[:-1] <= src[1:]))}")

In [ ]:
n = len(page_ids)

# Are the page_ids sorted and unique? Then index order = page_id order.
print("page_ids sorted & unique?", bool(np.all(np.diff(page_ids) > 0)))

# Lookup array: id_to_idx[page_id] -> dense index, or -1 if not an article
id_to_idx = np.full(page_ids.max() + 1, -1, dtype=np.int32)
id_to_idx[page_ids] = np.arange(n, dtype=np.int32)

src_idx = id_to_idx[src]
tgt_idx = id_to_idx[tgt]

print("every edge endpoint is an article?", bool((src_idx >= 0).all() and (tgt_idx >= 0).all()))
print("src_idx still sorted?", bool(np.all(src_idx[:-1] <= src_idx[1:])))
print(f"lookup array: {id_to_idx.nbytes / 1e6:.1f} MB")

In [ ]:
# 1. Out-degree: how many edges start at each index
out_deg = np.bincount(src_idx, minlength=n)

# 2.+3. offsets[0] = 0, offsets[i+1] = offsets[i] + out_deg[i]
offsets = np.zeros(n + 1, dtype=np.int32)
np.cumsum(out_deg, out=offsets[1:])

# targets: already grouped by source, because the edges were sorted
targets = tgt_idx

print("offsets[-1] == number of edges?", bool(offsets[-1] == len(targets)))
print(f"targets: {targets.nbytes / 1e6:.1f} MB   offsets: {offsets.nbytes / 1e6:.1f} MB")
print("articles with no outgoing links:", int((out_deg == 0).sum()))
print("largest out-degree:", int(out_deg.max()))

In [ ]:
titles = pages["title"].to_pylist()
title_to_idx = {t: i for i, t in enumerate(titles)}


def neighbours(i):
    return targets[offsets[i] : offsets[i + 1]]


i = title_to_idx["Munich"]
print("out-degree:", int(out_deg[i]))
print([titles[j] for j in neighbours(i)[:15]])

In [ ]:
import time
import tracemalloc

# --- CSR ---
t0 = time.perf_counter()
out_deg = np.bincount(src_idx, minlength=n)
offsets = np.zeros(n + 1, dtype=np.int64)
np.cumsum(out_deg, out=offsets[1:])
targets = tgt_idx
csr_time = time.perf_counter() - t0
csr_mb = (offsets.nbytes + targets.nbytes) / 1e6

# --- dict of lists ---
tracemalloc.start()
t0 = time.perf_counter()
adj = {i: [] for i in range(n)}
for s, t in zip(src_idx.tolist(), tgt_idx.tolist(), strict=False):
    adj[s].append(t)
dict_time = time.perf_counter() - t0
dict_mb = tracemalloc.get_traced_memory()[0] / 1e6
tracemalloc.stop()

print(f"CSR:           {csr_time:6.2f} s   {csr_mb:8.1f} MB")
print(f"dict of lists: {dict_time:6.2f} s   {dict_mb:8.1f} MB")
print("same neighbours for Munich?", adj[i] == neighbours(i).tolist())

In [ ]:
del adj